# Backtesting: Opening Range Breakout (ORB) Gap Trade Strategy

## Overview

We test a family of Opening Range Breakout strategies on gap trades that meet
the significance threshold (670 events over 8 years). The backtest explores:

- **Entry**: Breakout of first candle's range at 1, 5, or 30 minutes after open
- **Direction**: Only gap-up (long) or gap-down (short)
- **Stop Loss**: Day low or trailing stop
- **Exit**: Trailing 5/10-day SMA crossover or profit-taking %
- **Data**: 1-minute intraday or daily OHLC


---

## Test Matrix

| Entry Min | SL Type   | SL Value | Exit Type   | Exit Param  |
|-----------|-----------|----------|-------------|-------------|
| 1         | day_low   | —        | SMA_10      | —           |
| 1         | trailing  | 2%       | SMA_10      | —           |
| 5         | day_low   | —        | SMA_5       | —           |
| 5         | trailing  | 3%       | SMA_5       | —           |
| 30        | day_low   | —        | SMA_10      | —           |
| 30        | trailing  | 2%       | profit_pct  | 5%          |

Parallel runs produce a comparison table ranked by Sharpe or Win Rate.


In [2]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "config.py").exists():
    root = root.parent
sys.path.insert(0, str(root))  # Add the root directory to the Python path
sys.path.insert(0, str(root / "scripts"))  # Add the scripts directory to the Python path

from config import ROOT_DIR, MAIN_DIR
from backtester import BacktestConfig, ORBBacktester
import pandas as pd

print(f"Project root: {ROOT_DIR}")

Project root: C:\Users\j\python\Shocks_analysis_project


## Baseline Strategy: All Gap-Ups (No Earnings Filter)

Load baseline strategy candidates (all significant gap-ups regardless of earnings surprise) and backtest the Opening Range Breakout strategy across different entry times.

In [3]:
# Load quartile-assigned data with baseline strategy
merged_q = pd.read_csv(MAIN_DIR / "gap_earnings_with_quartiles.csv", parse_dates=["Date"])
baseline_long = merged_q[merged_q["Is_Gap_Up"] == True].copy()

print(f"Baseline strategy candidates (all gap-ups): {len(baseline_long)}")
print(f"Date range: {baseline_long['Date'].min()} to {baseline_long['Date'].max()}")


Baseline strategy candidates (all gap-ups): 1145
Date range: 2016-01-21 00:00:00 to 2025-02-20 00:00:00


### Baseline Test 1: Entry @ 1 min, Day-Low SL, SMA-10 Exit

In [ ]:
baseline_config_1 = BacktestConfig(
    entry_minutesAfterOpen=1,
    entry_type="gap_up",
    stoploss_type="day_low",
    stoploss_value=None,
    exit_type="trailing_sma",
    exit_param=10,
    profit_taking_levels=None,
    significance_filter="Sig30_P1",  # Use significant gaps only
    use_intraday=True
)

bt_base_1 = ORBBacktester(baseline_config_1)
trades_base_1 = bt_base_1.run()
metrics_base_1 = bt_base_1.compute_metrics(trades_base_1)

print("Baseline Test 1 (1 min entry) Results:")
print(f"  Total trades: {metrics_base_1['Total_Trades']:.0f}")
print(f"  Win rate: {metrics_base_1['Win_Rate_Pct']:.2%}")
print(f"  Avg trade: {metrics_base_1['Avg_Win_Pct']:.2f}")
print(f"  Profit factor: {metrics_base_1['Profit_Factor']:.2f}")
print(f"  Sharpe ratio: {metrics_base_1['Sharpe']:.2f}")
print(f"  Max drawdown: {metrics_base_1['Max_Drawdown_Pct']:.2%}")

### Baseline Test 2: Entry @ 5 min, Day-Low SL, SMA-10 Exit

In [ ]:
baseline_config_5 = BacktestConfig(
    entry_minutesAfterOpen=5,
    entry_type="gap_up",
    stoploss_type="day_low",
    stoploss_value=None,
    exit_type="trailing_sma",
    exit_param=10,
    profit_taking_levels=None,
    significance_filter=None,
    use_intraday=True
)

bt_base_5 = ORBBacktester(baseline_config_5)
trades_base_5 = bt_base_5.run()
metrics_base_5 = bt_base_5.compute_metrics(trades_base_5)

print("Baseline Test 2 (5 min entry) Results:")
print(f"  Total trades: {metrics_base_5['total_trades']:.0f}")
print(f"  Win rate: {metrics_base_5['win_rate']:.2%}")
print(f"  Avg trade: {metrics_base_5['avg_trade']:.2f}")
print(f"  Profit factor: {metrics_base_5['profit_factor']:.2f}")
print(f"  Sharpe ratio: {metrics_base_5['sharpe_ratio']:.2f}")
print(f"  Max drawdown: {metrics_base_5['max_drawdown']:.2%}")


### Baseline Test 3: Entry @ 30 min, Day-Low SL, SMA-10 Exit

In [ ]:
baseline_config_30 = BacktestConfig(
    entry_minutesAfterOpen=30,
    entry_type="gap_up",
    stoploss_type="day_low",
    stoploss_value=None,
    exit_type="trailing_sma",
    exit_param=10,
    profit_taking_levels=None,
    significance_filter=None,
    use_intraday=True
)

bt_base_30 = ORBBacktester(baseline_config_30)
trades_base_30 = bt_base_30.run()
metrics_base_30 = bt_base_30.compute_metrics(trades_base_30)

print("Baseline Test 3 (30 min entry) Results:")
print(f"  Total trades: {metrics_base_30['total_trades']:.0f}")
print(f"  Win rate: {metrics_base_30['win_rate']:.2%}")
print(f"  Avg trade: {metrics_base_30['avg_trade']:.2f}")
print(f"  Profit factor: {metrics_base_30['profit_factor']:.2f}")
print(f"  Sharpe ratio: {metrics_base_30['sharpe_ratio']:.2f}")
print(f"  Max drawdown: {metrics_base_30['max_drawdown']:.2%}")


### Baseline Summary: Entry Timing Comparison

In [ ]:
# Create baseline comparison table
baseline_comparison = pd.DataFrame({
    "Entry (min)": [1, 5, 30],
    "Total Trades": [
        metrics_base_1['total_trades'],
        metrics_base_5['total_trades'],
        metrics_base_30['total_trades']
    ],
    "Win Rate": [
        metrics_base_1['win_rate'],
        metrics_base_5['win_rate'],
        metrics_base_30['win_rate']
    ],
    "Avg Trade ($)": [
        metrics_base_1['avg_trade'],
        metrics_base_5['avg_trade'],
        metrics_base_30['avg_trade']
    ],
    "Sharpe Ratio": [
        metrics_base_1['sharpe_ratio'],
        metrics_base_5['sharpe_ratio'],
        metrics_base_30['sharpe_ratio']
    ],
    "Max Drawdown": [
        metrics_base_1['max_drawdown'],
        metrics_base_5['max_drawdown'],
        metrics_base_30['max_drawdown']
    ],
    "Profit Factor": [
        metrics_base_1['profit_factor'],
        metrics_base_5['profit_factor'],
        metrics_base_30['profit_factor']
    ]
})

print("BASELINE STRATEGY: All Gap-Ups, Day-Low SL, SMA-10 Exit")
print(baseline_comparison.to_string(index=False))


## Test 1: ORB @ 1 min, Day-Low SL, 10-day SMA Exit

In [ ]:
config1 = BacktestConfig(
    entry_minutesAfterOpen=1,
    entry_type="gap_up",
    stoploss_type="day_low",
    stoploss_value=None,
    exit_type="trailing_sma",
    exit_param=10,
    profit_taking_levels=None,
    significance_filter="Sig30_P5",
    use_intraday=True
)

bt1 = ORBBacktester(config1)
trades1 = bt1.run()
metrics1 = bt1.compute_metrics(trades1)

print("Test 1 Results:")
for k, v in metrics1.items():
    print(f"  {k}: {v:.2f}")

trades1.head()

## Test 2: ORB @ 1 min, Trailing 2% SL, 10-day SMA Exit

In [9]:
config2 = BacktestConfig(
    entry_minutesAfterOpen=1,
    entry_type="gap_up",
    stoploss_type="trailing",
    stoploss_value=0.02,
    exit_type="trailing_sma",
    exit_param=10,
    profit_taking_levels=None,
    significance_filter="Sig30_P5",
    use_intraday=True
)

bt2 = ORBBacktester(config2)
trades2 = bt2.run()
metrics2 = bt2.compute_metrics(trades2)

print("Test 2 Results:")
for k, v in metrics2.items():
    print(f"  {k}: {v:.2f}")

trades2.head()

Loaded 13475 gap trades (type=gap_up, filter=Sig30_P5)
Test 2 Results:
  Total_Trades: 5642.00
  Winning_Trades: 436.00
  Losing_Trades: 5204.00
  Win_Rate_Pct: 7.73
  Total_Return_Pct: -2898.82
  Avg_Win_Pct: 17.15
  Avg_Loss_Pct: -1.99
  Profit_Factor: 0.72
  Sharpe: -0.94
  Max_Drawdown_Pct: -100.00


,Ticker,Entry_Date,Entry_Time,Entry_Price,Exit_Date,Exit_Time,Exit_Price,Exit_Reason,Direction,PnL,PnL_Pct,Max_Unrealized_Gain_Pct,Max_Unrealized_Loss_Pct
0,AACG,2017-08-30,09:31:00,5.22,2017-08-30,09:40:00,5.1156,SL,long,-0.1044,-0.020000,0.003831,-0.065134
1,AACG,2019-04-09,09:31:00,1.17,2019-04-09,19:59:00,2.8400,EOD,long,1.6700,1.427350,1.632479,-0.017094
2,AACG,2021-02-04,09:31:00,3.77,2021-02-04,19:59:00,10.0900,EOD,long,6.3200,1.676393,4.238727,-0.005305
3,AACG,2023-03-07,09:31:00,2.12,2023-03-07,10:50:00,2.0776,SL,long,-0.0424,-0.020000,0.146226,-0.028302
4,AAL,2021-01-28,09:31:00,20.40,2021-01-28,09:57:00,19.9920,SL,long,-0.4080,-0.020000,0.067157,-0.029412


## Test 3: ORB @ 5 min, Day-Low SL, 5-day SMA Exit

In [7]:
config3 = BacktestConfig(
    entry_minutesAfterOpen=5,
    entry_type="gap_up",
    stoploss_type="day_low",
    stoploss_value=None,
    exit_type="trailing_sma",
    exit_param=5,
    profit_taking_levels=None,
    significance_filter="Sig30_P5",
    use_intraday=True
)

bt3 = ORBBacktester(config3)
trades3 = bt3.run()
metrics3 = bt3.compute_metrics(trades3)

print("Test 3 Results:")
for k, v in metrics3.items():
    print(f"  {k}: {v:.2f}")

trades3.head()

Loaded 13475 gap trades (type=gap_up, filter=Sig30_P5)
Test 3 Results:
  Total_Trades: 5226.00
  Winning_Trades: 1596.00
  Losing_Trades: 3612.00
  Win_Rate_Pct: 30.54
  Total_Return_Pct: -33335.44
  Avg_Win_Pct: 21.45
  Avg_Loss_Pct: -18.71
  Profit_Factor: 0.51
  Sharpe: -3.21
  Max_Drawdown_Pct: -100.00


,Ticker,Entry_Date,Entry_Time,Entry_Price,Exit_Date,Exit_Time,Exit_Price,Exit_Reason,Direction,PnL,PnL_Pct,Max_Unrealized_Gain_Pct,Max_Unrealized_Loss_Pct
0,AACG,2019-04-09,09:35:00,1.5900,2019-04-09,19:59:00,2.84,EOD,long,1.2500,0.786164,0.937107,-0.050314
1,AACG,2021-02-04,09:35:00,4.6578,2021-02-04,19:59:00,10.09,EOD,long,5.4322,1.166259,3.240199,-0.139079
2,AACG,2023-03-07,09:35:00,2.2199,2023-03-07,19:09:00,2.25,EOD,long,0.0301,0.013559,0.094644,-0.072030
3,AAL,2020-06-05,09:35:00,21.8350,2020-06-05,19:59:00,19.09,EOD,long,-2.7450,-0.125716,0.044195,-0.184795
4,AAL,2021-01-28,09:35:00,21.4600,2021-01-28,19:59:00,18.94,EOD,long,-2.5200,-0.117428,0.014445,-0.216216


## Test 4: ORB @ 5 min, Trailing 3% SL, 5-day SMA Exit

In [8]:
config4 = BacktestConfig(
    entry_minutesAfterOpen=5,
    entry_type="gap_up",
    stoploss_type="trailing",
    stoploss_value=0.03,
    exit_type="trailing_sma",
    exit_param=5,
    profit_taking_levels=None,
    significance_filter="Sig30_P5",
    use_intraday=True
)

bt4 = ORBBacktester(config4)
trades4 = bt4.run()
metrics4 = bt4.compute_metrics(trades4)

print("Test 4 Results:")
for k, v in metrics4.items():
    print(f"  {k}: {v:.2f}")

trades4.head()

Loaded 13475 gap trades (type=gap_up, filter=Sig30_P5)
Test 4 Results:
  Total_Trades: 5226.00
  Winning_Trades: 518.00
  Losing_Trades: 4705.00
  Win_Rate_Pct: 9.91
  Total_Return_Pct: -3811.46
  Avg_Win_Pct: 19.69
  Avg_Loss_Pct: -2.98
  Profit_Factor: 0.73
  Sharpe: -0.81
  Max_Drawdown_Pct: -100.00


,Ticker,Entry_Date,Entry_Time,Entry_Price,Exit_Date,Exit_Time,Exit_Price,Exit_Reason,Direction,PnL,PnL_Pct,Max_Unrealized_Gain_Pct,Max_Unrealized_Loss_Pct
0,AACG,2019-04-09,09:35:00,1.5900,2019-04-09,09:35:00,1.542300,SL,long,-0.047700,-0.03,0.106918,-0.050314
1,AACG,2021-02-04,09:35:00,4.6578,2021-02-04,09:35:00,4.518066,SL,long,-0.139734,-0.03,0.021942,-0.055348
2,AACG,2023-03-07,09:35:00,2.2199,2023-03-07,10:49:00,2.153303,SL,long,-0.066597,-0.03,0.094644,-0.040317
3,AAL,2020-06-05,09:35:00,21.8350,2020-06-05,09:36:00,21.179950,SL,long,-0.655050,-0.03,0.000229,-0.033662
4,AAL,2021-01-28,09:35:00,21.4600,2021-01-28,09:38:00,20.816200,SL,long,-0.643800,-0.03,0.014445,-0.039581


## Test 5: ORB @ 30 min, Day-Low SL, 10-day SMA Exit

In [ ]:
config5 = BacktestConfig(
    entry_minutesAfterOpen=30,
    entry_type="gap_up",
    stoploss_type="day_low",
    stoploss_value=None,
    exit_type="trailing_sma",
    exit_param=10,
    profit_taking_levels=None,
    significance_filter="Sig30_P5",
    use_intraday=True
)

bt5 = ORBBacktester(config5)
trades5 = bt5.run()
metrics5 = bt5.compute_metrics(trades5)

print("Test 5 Results:")
for k, v in metrics5.items():
    print(f"  {k}: {v:.2f}")

trades5.head()

## Test 6: ORB @ 30 min, Trailing 2% SL, 5% Profit-Taking

In [ ]:
config6 = BacktestConfig(
    entry_minutesAfterOpen=30,
    entry_type="gap_up",
    stoploss_type="trailing",
    stoploss_value=0.02,
    exit_type="profit_pct",
    exit_param=0.05,
    profit_taking_levels=None,
    significance_filter="Sig30_P5",
    use_intraday=True
)

bt6 = ORBBacktester(config6)
trades6 = bt6.run()
metrics6 = bt6.compute_metrics(trades6)

print("Test 6 Results:")
for k, v in metrics6.items():
    print(f"  {k}: {v:.2f}")

trades6.head()

## Summary: Comparison of All Tests

Rank by Sharpe ratio and Win Rate

In [6]:
summary_df = pd.DataFrame([
    {"Test": 1, "Entry Min": 1, "SL": "day_low", "SL Value": "—", 
     "Exit": "SMA_10", **metrics1},
    {"Test": 2, "Entry Min": 1, "SL": "trailing", "SL Value": "2%", 
     "Exit": "SMA_10", **metrics2},
    {"Test": 3, "Entry Min": 5, "SL": "day_low", "SL Value": "—", 
     "Exit": "SMA_5", **metrics3},
    {"Test": 4, "Entry Min": 5, "SL": "trailing", "SL Value": "3%", 
     "Exit": "SMA_5", **metrics4},
    {"Test": 5, "Entry Min": 30, "SL": "day_low", "SL Value": "—", 
     "Exit": "SMA_10", **metrics5},
    {"Test": 6, "Entry Min": 30, "SL": "trailing", "SL Value": "2%", 
     "Exit": "PT_5%", **metrics6},
])

# Rank by Sharpe
summary_by_sharpe = summary_df.sort_values("Sharpe", ascending=False)
print("\n=== Ranked by Sharpe Ratio ===")
print(summary_by_sharpe[["Test", "Entry Min", "SL", "Exit", "Sharpe", "Win_Rate_Pct", "Total_Return_Pct", "Total_Trades"]])

# Rank by Win Rate
summary_by_wr = summary_df.sort_values("Win_Rate_Pct", ascending=False)
print("\n=== Ranked by Win Rate ===")
print(summary_by_wr[["Test", "Entry Min", "SL", "Exit", "Win_Rate_Pct", "Sharpe", "Total_Return_Pct"]])

# Save full summary
summary_df.to_csv(MAIN_DIR / "backtest_summary.csv", index=False)
print(f"\nFull summary saved to {MAIN_DIR}/backtest_summary.csv")

NameError: name 'metrics1' is not defined

## Export Trade Logs for Analysis

Save each test's trades for further inspection or walk-forward analysis

In [ ]:
trades_dict = {
    "Test_1_ORB1_DayLow_SMA10": trades1,
    "Test_2_ORB1_Trail2_SMA10": trades2,
    "Test_3_ORB5_DayLow_SMA5": trades3,
    "Test_4_ORB5_Trail3_SMA5": trades4,
    "Test_5_ORB30_DayLow_SMA10": trades5,
    "Test_6_ORB30_Trail2_PT5": trades6,
}

backtest_dir = MAIN_DIR / "backtest_results"
backtest_dir.mkdir(exist_ok=True)

for name, df in trades_dict.items():
    csv_path = backtest_dir / f"{name}.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved {name}: {len(df)} trades → {csv_path}")